# C5-neural-networks — Session 3: Initialization and Variance

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2 (the
MLP forward pass, `affine_layer`) and F5-probability Session 2 (variance,
independence, the scaled-sums identity
$\operatorname{Var}[\sum_k w_k x_k] = \sigma^2 \sum_k w_k^2$).*

**This session:** the one place in this unit where weights are *random*
rather than designed — and the single number you must choose about them:
their **scale**.
When a layer is too wide for geometric insight (say 100 inputs), you draw
its weights at random; pick the spread badly and the forward pass destroys
itself within a few layers, activations exploding to $10^{20}$ or dying to
$10^{-10}$.
Today: watch that happen, then *derive* the fix — F5's variance toolkit,
extended by one stated fact, gives the exam's favorite init rule
$\sigma = 1/\sqrt{C}$ — and verify the derivation by seeded simulation.
Everything here is about keeping the **forward pass** healthy; no training
appears anywhere (Session 1's scope note stands).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804


def affine_layer(x, W, b):
    """Pre-activations z[i, j] = sum_k x[i, k] * W[j, k] + b[j]."""
    return (x[:, None, :] * W[None, :, :]).sum(axis=2) + b

## 1. The Problem: Activation Scales in a Deep Stack

**The experiment first.**
Take a stack of $D = 30$ layers, each $100 \to 100$, biases $0$, no
activation in between (the cleanest scale bookkeeping; activations return
in §6).
Feed in a batch of standard-normal inputs and draw every weight
independently from $N(0, \sigma^2)$, for three choices of $\sigma$.
The only thing we track is the **spread of the values** flowing through —
the standard deviation of the layer outputs:

In [ ]:
C, D, n_batch = 100, 30, 200
report_at = (1, 5, 10, 20, 30)

for sigma in (0.05, 0.1, 0.5):
    rng = np.random.default_rng(SEED)
    h = rng.normal(0, 1, (n_batch, C))
    stds = []
    for _ in range(D):
        W = rng.normal(0, sigma, (C, C))
        h = affine_layer(h, W, np.zeros(C))
        stds.append(h.std())
    row = "   ".join(f"L{t}: {stds[t - 1]:9.3g}" for t in report_at)
    print(f"sigma = {sigma:4}   " + row)

Read the three rows:

- $\sigma = 0.05$: the spread is cut roughly in **half every layer** —
  $0.501$ after one layer, $8.5 \times 10^{-10}$ after thirty.
  The signal *vanishes*: by layer 30 every unit outputs essentially 0, and
  whatever the input was is numerically gone.
- $\sigma = 0.5$: the spread **quintuples every layer** — $5.01$, then
  $8.5 \times 10^{20}$ by layer 30. The signal *explodes* toward overflow.
- $\sigma = 0.1$: the spread hovers near $1$ all the way down
  ($1.0 \to 0.909$) — a healthy pipe.

(The three runs share one seed, so they see proportionally identical weight
draws — the *only* difference is the scale $\sigma$, which is exactly the
point.)
Per layer, the three multipliers are about $0.5$, $1$, and $5$.
The goal of this session is to predict that multiplier from $\sigma$ and
the width $C$, and then choose $\sigma$ to pin it at $1$.

### Checkpoint 1

1. From the printed table: estimate the per-layer factor for
   $\sigma = 0.05$, and predict the spread after 40 layers.
2. A 12-layer stack multiplies the spread by $0.5$ per layer.
   Roughly what spread does a unit-spread input have at the output, and
   why is that a problem for whatever reads it?

## 2. One Pre-Activation Is a Scaled Sum — with Random Scales

Focus on a single unit in the first layer: its pre-activation on one input
point is

$$z \;=\; \sum_{k=1}^{C} W_k\, x_k
\qquad\text{(bias set to 0 at initialization, the pinned convention).}$$

**Modeling assumptions, stated once and used all session:**

- the inputs $x_1, \dots, x_C$ are random variables, mutually independent,
  each with mean $0$ and variance $1$ (standardized features — C4's
  scaling discipline feeding this unit);
- the weights $W_1, \dots, W_C$ are drawn mutually independently from a
  distribution with mean $0$ and variance $\sigma^2$;
- all weights are independent of all inputs.

F5 computed the variance of exactly this shape of sum — *when the scale
factors were constants*:
$\operatorname{Var}[\sum_k w_k x_k] = \sum_k w_k^2 \cdot 1$.
Here the scales are **themselves random**, so that identity does not apply
as stated: the summands are now *products of two random variables*,
$W_k x_k$, and we need their variance and their independence before
variance-of-sums can fire.
One stated fact closes the gap.

### Checkpoint 2

1. With *constant* weights $w = (0.3, -0.4, 1.2)$ and independent
   unit-variance, mean-0 inputs: compute $\operatorname{Var}[z]$ from F5's
   identity.
2. Point to the exact step of F5's identity that breaks when the $w_k$
   are random. One sentence.

## 3. The Group-Form Independence Fact, and the Variance of One Product

**Stated fact (group form; proof beyond this course, simulation check
below).**
F5 stated the pairwise form: if $X \perp Y$, then $g(X) \perp h(Y)$ for any
functions $g, h$.
The same holds for **groups**: *if a collection of random variables is
mutually independent, then random variables computed from disjoint
sub-collections are mutually independent.*
In particular, with $W_1, \dots, W_C, x_1, \dots, x_C$ all mutually
independent, the products

$$W_1 x_1,\; W_2 x_2,\; \dots,\; W_C x_C$$

are mutually independent — each is computed from its own disjoint pair
$(W_k, x_k)$.

**The variance of one product** — new here, and the exam's favorite
sub-step.
For one term $P = W x$ with $W \perp x$, both mean 0,
$\operatorname{Var}[W] = \sigma^2$, $\operatorname{Var}[x] = 1$:

- *Mean:* $E[P] = E[W]\,E[x] = 0$ (F5's product rule for independent
  variables).
- *Variance:* with zero mean, $\operatorname{Var}[P] = E[P^2] = E[W^2 x^2]$.
  Now $W^2$ and $x^2$ are functions of $W$ and $x$ separately, so they are
  independent (F5's pairwise fact), and the product rule applies again:
  $E[W^2 x^2] = E[W^2]\, E[x^2]$.
  Finally $E[W^2] = \operatorname{Var}[W] = \sigma^2$ and
  $E[x^2] = \operatorname{Var}[x] = 1$ (zero-mean shortcut, both times):

$$\boxed{\;\operatorname{Var}[W x] = \sigma^2 \cdot 1 = \sigma^2.\;}$$

Both claims, checked by seeded simulation with $\sigma = 0.5$
(so $\operatorname{Var}[Wx]$ should be $0.25$):

In [ ]:
rng = np.random.default_rng(SEED)
n = 200_000
W_draws = rng.normal(0, 0.5, n)
x_draws = rng.normal(0, 1, n)
P = W_draws * x_draws
print("E[Wx] estimate  :", f"{P.mean():.4f}", "   (exact 0)")
print("Var[Wx] estimate:", f"{P.var():.4f}", "   (exact 0.25)")

Estimates $0.0006$ and $0.2506$ — both within simulation noise of the
derived values.

**Now the whole pre-activation.**
The products $W_k x_k$ are mutually independent (group-form fact), so F5's
variance-of-sums applies term by term:

$$\operatorname{Var}[z]
= \operatorname{Var}\Big[\sum_{k=1}^{C} W_k x_k\Big]
= \sum_{k=1}^{C} \operatorname{Var}[W_k x_k]
= \sum_{k=1}^{C} \sigma^2
= C\,\sigma^2 .$$

Check: $C = 100$, $\sigma = 0.2$ predicts $\operatorname{Var}[z] = 4$:

In [ ]:
rng = np.random.default_rng(SEED)
n, C_demo = 100_000, 100
W_mat = rng.normal(0, 0.2, (n, C_demo))     # fresh weights every sample
x_mat = rng.normal(0, 1, (n, C_demo))       # fresh inputs every sample
z_samples = (W_mat * x_mat).sum(axis=1)
print("E[z] estimate  :", f"{z_samples.mean():.4f}", "   (exact 0)")
print("Var[z] estimate:", f"{z_samples.var():.4f}", "   (exact C sigma^2 = 4)")

`-0.0038` and `3.9844` — the formula holds.

### Checkpoint 3

1. The derivation of $\operatorname{Var}[z] = C\sigma^2$ uses three facts:
   the group-form fact, the product rule for independent variables, and
   variance-of-sums. Say which step uses which.
2. $C = 50$, $\sigma = 0.3$: compute $\operatorname{Var}[z]$ and the
   standard deviation of $z$.
3. If the weights had mean $\mu \ne 0$ (variance still $\sigma^2$), what is
   $E[W^2]$ — and is it still $\operatorname{Var}[W]$?

## 4. The $1/\sqrt{C}$ Rule

**The design goal — a forward-pass statement.**
The inputs arrive with variance 1.
If each pre-activation also has variance 1, the layer's outputs are on the
same scale as its inputs, the *next* layer faces the same healthy situation,
and the pipe stays healthy at any depth: **stable activation scales**.
So demand

$$\operatorname{Var}[z] = C \sigma^2 \overset{!}{=} 1
\qquad\Longleftrightarrow\qquad
\sigma^2 = \frac{1}{C}
\qquad\Longleftrightarrow\qquad
\boxed{\;\sigma = \frac{1}{\sqrt{C}}\;}$$

— *scale random weights by one over the square root of the number of
inputs they sum over.*

**This also explains §1's multipliers.**
If a layer's inputs have variance $v$ (not necessarily 1), the same
derivation gives $\operatorname{Var}[z] = C\sigma^2 v$: each layer
multiplies the variance by $C\sigma^2$, hence the standard deviation by
$\sigma\sqrt{C}$.
For §1's stack ($C = 100$): $\sigma = 0.05 \Rightarrow$ factor $0.5$;
$\sigma = 0.1 = 1/\sqrt{100} \Rightarrow$ factor $1$;
$\sigma = 0.5 \Rightarrow$ factor $5$ — precisely the three multipliers
observed.

**The rule at three widths**, 20 layers deep, seeded:

In [ ]:
for C_w in (25, 100, 400):
    rng = np.random.default_rng(SEED)
    h = rng.normal(0, 1, (200, C_w))
    for _ in range(20):
        W = rng.normal(0, 1 / np.sqrt(C_w), (C_w, C_w))
        h = affine_layer(h, W, np.zeros(C_w))
    print(f"C = {C_w:3d}, sigma = 1/sqrt(C): spread after 20 layers = {h.std():.4f}")

All three runs end with spread of order 1 — $1.3076$, $0.9090$, $0.9480$
— after twenty layers.
Not exactly $1.0000$, and honestly so: the derivation pins the variance of
the *distribution* each layer draws from; a single random run drifts around
that value, and narrower layers (fewer terms averaging) drift more — the
$C = 25$ run wanders farthest.
Compare §1: without the rule the same twenty layers moved the scale by ten
orders of magnitude either way.

### Checkpoint 4

1. A layer sums over $C = 900$ inputs. What $\sigma$ does the rule
   prescribe?
2. $C = 50$, $\sigma = 0.2$: what factor does each layer apply to the
   *standard deviation*, and is the stack growing or shrinking?
3. For which width $C$ is $\sigma = 0.05$ exactly the rule's choice?

## 5. Worked Exam-Style Example 3: the Init Derivation, Graded Register

The exam asks this as a *reasoning-required* derivation: state the facts,
chain them, produce an exact number.
Here is one solved in that register.

---

**Problem (reasoning is required).**
A hidden layer receives $C = 64$ inputs, mutually independent, each with
mean $0$ and variance $1$.
Its weights are drawn mutually independently, mean $0$, standard deviation
$\sigma > 0$, independent of all inputs; the bias is $0$.
Derive the exact value of $\sigma$ for which each pre-activation
$z = \sum_{k=1}^{64} W_k x_k$ has $\operatorname{Var}[z] = 1$.
Give $\sigma$ as an exact fraction.

---

**Solution, claim by claim.**

1. *The summands are independent.* Each product $W_k x_k$ is computed from
   the disjoint pair $(W_k, x_k)$ of the mutually independent collection
   $\{W_1, \dots, W_{64}, x_1, \dots, x_{64}\}$ — by the group-form
   independence fact, the 64 products are mutually independent.
2. *Each summand's variance.* $E[W_k x_k] = E[W_k]E[x_k] = 0$, so
   $\operatorname{Var}[W_k x_k] = E[W_k^2 x_k^2] = E[W_k^2]\,E[x_k^2]
   = \sigma^2 \cdot 1$ (independence of $W_k^2$ and $x_k^2$, then the
   zero-mean shortcut twice).
3. *Sum.* Variance-of-sums over independent terms:
   $\operatorname{Var}[z] = 64\,\sigma^2$.
4. *Solve.* $64 \sigma^2 = 1$ and $\sigma > 0$ give
   $\sigma = \sqrt{1/64} = \mathbf{1/8}$.

Simulation check at $\sigma = 1/8$ (variance estimate should be near 1):

In [ ]:
rng = np.random.default_rng(SEED)
n = 100_000
W_mat = rng.normal(0, 1 / 8, (n, 64))
x_mat = rng.normal(0, 1, (n, 64))
z_samples = (W_mat * x_mat).sum(axis=1)
print("Var[z] estimate at sigma = 1/8:", f"{z_samples.var():.4f}", "   (exact 1)")

`0.9950` — the derivation and the simulation agree.
Grading note (this is how p12's rubric works too): each numbered claim is
worth points on its own, and naming the fact that licenses the step — group
form, product rule, variance-of-sums — is part of the answer.

### Checkpoint 5

1. Same problem but the inputs have variance $4$ and the target is
   $\operatorname{Var}[z] = 4$, with $C = 25$: derive $\sigma$.
2. Same problem, $C = 64$, but the answer is requested as $\sigma^2$ in
   lowest terms $p/q$ — what are $p$, $q$, and $p + q$ (normal-form
   decoding)?

## 6. Common Pitfalls III

**Pitfall 7 — scaling by $1/C$ instead of $1/\sqrt{C}$.**
The rule lives on the *variance* ($\sigma^2 = 1/C$); pulling the $1/C$
onto $\sigma$ itself is the classic square-root slip, and it is fatal:
$\sigma = 1/C$ gives per-layer factor
$\sigma\sqrt{C} = 1/\sqrt{C} = 0.1$ at $C = 100$ — a *tenfold shrink per
layer*:

In [ ]:
rng = np.random.default_rng(SEED)
h = rng.normal(0, 1, (200, 100))
for _ in range(10):
    W = rng.normal(0, 1 / 100, (100, 100))     # BROKEN: sigma = 1/C
    h = affine_layer(h, W, np.zeros(100))
print("sigma = 1/C, spread after 10 layers:", f"{h.std():.3g}", "   (prediction: 1e-10)")

`9.13e-11` after just ten layers — against the predicted
$0.1^{10} = 10^{-10}$.
If your activations die but your formula "has a $1/C$ in it", check which
side of the square root the $C$ sits on.

**Pitfall 8 — nonzero-mean weights: $E[W^2] \ne \operatorname{Var}[W]$.**
The derivation used $E[W^2] = \sigma^2$, which is true *only for mean-zero
weights*.
With mean $\mu$: $E[W^2] = \mu^2 + \sigma^2$, so
$\operatorname{Var}[z] = C(\mu^2 + \sigma^2)$.
Weights drawn with $\mu = 0.2$, $\sigma = 0.1$ at $C = 100$ predict
variance $100 \cdot (0.04 + 0.01) = 5$, not $1$:

In [ ]:
rng = np.random.default_rng(SEED)
n = 100_000
W_mat = rng.normal(0.2, 0.1, (n, 100))       # BROKEN init: mean 0.2
x_mat = rng.normal(0, 1, (n, 100))
z_samples = (W_mat * x_mat).sum(axis=1)
print("Var[z] estimate with mean-0.2 weights:", f"{z_samples.var():.4f}", "   (prediction 5)")

`4.9738` — five times the intended scale, from a mean shift alone.
Mean-zero initialization is a working assumption of every formula in this
session, not a stylistic choice.

**Pitfall 9 — expecting the identity-stack bookkeeping to survive a
nonlinearity unchanged.**
§1–§4 tracked an activation-free stack on purpose: it isolates the scale
arithmetic.
Insert a ReLU after every layer and half of each layer's output is zeroed —
the spread now shrinks by roughly $1/\sqrt{2}$ per layer even at
$\sigma = 1/\sqrt{C}$:

In [ ]:
rng = np.random.default_rng(SEED)
h = rng.normal(0, 1, (200, 100))
for _ in range(20):
    W = rng.normal(0, 0.1, (100, 100))          # the rule's sigma for C = 100 ...
    h = np.maximum(affine_layer(h, W, np.zeros(100)), 0.0)   # ... but ReLU after every layer
print("ReLU stack, spread after 20 layers:", f"{h.std():.4g}",
      "   (roughly (1/sqrt(2))^20 = 1e-3)")

`0.0007265` after twenty layers — versus $0.909$ for the identity stack
with the same seed and $\sigma$.
The *method* — track the variance layer by layer, choose $\sigma$ to hold
it — is exactly right; the *constant* changes with the activation (ReLU
init multiplies the rule by $\sqrt{2}$; tanh has its own correction).
Those constants belong to later units; what this course grades is the
$1/\sqrt{C}$ derivation and the discipline behind it.

### Checkpoint 6

1. A teammate's 8-layer, width-64 stack has spreads
   $1, 0.125, 0.0156, \dots$ — a factor of exactly $1/8$ per layer.
   Which pitfall, and what was their $\sigma$?
2. Weights drawn from $N(0.5, 0.01)$ at $C = 100$: compute the predicted
   $\operatorname{Var}[z]$ (inputs standard).
3. True or false, one sentence: because of Pitfall 9, the variance method
   itself is wrong for ReLU networks.

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **PyTorch engineering cluster** (7 sub-parts, 50 points in r1-2026)
  is built around *inference-only* networks with **manually designed
  weights implementing geometric predicates** — exactly Session 2's
  half-plane-detector + gate pattern (p09, p13, p14, p17 train it, and
  C6-pytorch re-runs it through torch modules built on this unit's
  `affine_layer` / `step_activation` by name).
- The **probability/statistics sub-part on variance-preserving
  initialization** (5 points) is Session 3's derivation verbatim in
  spirit: state independence, compute a product variance, sum, solve for
  the scale (p10, p12, p16).
  The r2-2026 rationale material flags exactly this chain of *stated facts
  used precisely* as a discriminator.
- The **ML concepts cluster** (5 concept MCQs, 50 points) reaches
  activation properties and architecture reading at the register of
  p01–p03 — among the most accessible points on the paper.
- The paper's **multi-part arcs** feed earlier results into later parts
  (design → implement → verify → count); p13, p14, and p17 rehearse that
  texture, with p04's normal-form decoding as the hand-arithmetic atom.

## Going Deeper

Optional forward pointers along the course map — nothing here is needed
for this unit's practice:

- **`C6-pytorch`**: this unit's two helpers become `nn.Module` subclasses
  (`DenseLayer`, `ThresholdGate`) with registered parameters; the same
  hand-designed weights get loaded into torch tensors, and parameter
  counting/inspection becomes the new skill. The forward passes must agree
  with this unit's NumPy numbers exactly.
- **`C7-cnn-transfer`**: convolutional layers are affine layers with
  shared, spatially-slid weights; pretrained networks are "weights someone
  else already chose", pushing this unit's design-not-train stance to its
  industrial conclusion. The init corrections for ReLU stacks (the
  $\sqrt{2}$ of Pitfall 9) reappear there as the schemes real frameworks
  ship.
- **The boundary this course keeps:** backpropagation — the algorithm that
  would *train* these networks — stays outside the syllabus.
  Every exam problem in this cluster is inference: given or designed
  weights, executed exactly.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. About $0.5$ per layer (e.g. $0.501$ after one layer; each reported
   checkpoint is consistent with halving). After 40 layers:
   $\approx 0.5^{40} \approx 9 \times 10^{-13}$ — even deader than the
   30-layer value.
2. $0.5^{12} \approx 2.4 \times 10^{-4}$. Whatever reads the output must
   distinguish values crammed into a $\sim 10^{-4}$-wide band — float
   noise and any later thresholds swamp the signal.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\operatorname{Var}[z] = 0.09 + 0.16 + 1.44 = 1.69$.
2. F5's identity treats $w_k^2$ as constants that factor out of each
   term's variance ($\operatorname{Var}[w_k x_k] = w_k^2
   \operatorname{Var}[x_k]$); with random $W_k$ nothing constant factors
   out — the term is a product of two random variables.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Group form → the products $W_k x_k$ are mutually independent;
   product rule (for $W_k^2 \perp x_k^2$, plus $E[W_k x_k] = 0$) → each
   product's variance is $\sigma^2$; variance-of-sums → the variances add
   to $C\sigma^2$.
2. $\operatorname{Var}[z] = 50 \cdot 0.09 = 4.5$;
   $\operatorname{sd}[z] = \sqrt{4.5} \approx 2.121$.
3. $E[W^2] = \mu^2 + \sigma^2$ — no: $\operatorname{Var}[W] = E[W^2] -
   \mu^2$, so the two differ by exactly $\mu^2$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $\sigma = 1/\sqrt{900} = 1/30$.
2. Factor $\sigma\sqrt{C} = 0.2 \cdot \sqrt{50} \approx 1.414$ — growing
   (about $\sqrt 2$ per layer).
3. $1/\sqrt{C} = 0.05 \iff \sqrt{C} = 20 \iff C = 400$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $\operatorname{Var}[z] = C \sigma^2 \cdot
   \operatorname{Var}[x] = 25 \sigma^2 \cdot 4 = 100\sigma^2
   \overset{!}{=} 4$, so $\sigma^2 = 1/25$ and $\sigma = 1/5$.
2. $\sigma^2 = 1/64$: $p = 1$, $q = 64$, $p + q = 65$.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Pitfall 7 with a twist you can solve: the per-layer factor is
   $\sigma\sqrt{64} = 8\sigma = 1/8$, so $\sigma = 1/64 = 1/C$ — the
   square-root slip.
2. $\operatorname{Var}[z] = C(\mu^2 + \sigma^2) = 100(0.25 + 0.0001)
   = 25.01$.
3. False: the method (track variance, solve for scale) is exactly how the
   ReLU correction is derived too — only the constant multiplying the rule
   changes with the activation.

</details>